# 02. Cohort Retention & Customer Segmentation
### Track: Data Analytics (Customer Lifecycle Analysis)

**Tujuan Analisis:**
1. Memahami kurva retensi pelanggan berdasarkan kohort masa berlangganan (*Tenure Cohorts*).
2. Mengidentifikasi titik kritis (*Critical Drop-off Point*) di mana pelanggan paling rentan churn.
3. Membangun matriks risiko (*Hazard Matrix*) interaksi antara Tenure dan Tipe Kontrak.
4. Memetakan segmen pelanggan dengan risiko churn tertinggi untuk prioritas program retensi.


In [ ]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

PROJECT_ROOT = Path.cwd().resolve().parents[1] if Path.cwd().name == "01_analytics" else Path.cwd().resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from src.telco_churn.core.loader import load_processed_data, FIGURES_DIR
from src.telco_churn.analytics.aggregations import build_tenure_cohorts, build_risk_matrix

sns.set_theme(style="whitegrid", font="sans-serif")
plt.rcParams["figure.dpi"] = 120


In [ ]:
df = load_processed_data()
print(f"Memuat {len(df):,} baris data bersih.")
df.head(3)


## 1. Analisis Kohort Siklus Hidup Pelanggan (*Tenure Cohorts*)
Kita mengelompokkan pelanggan ke dalam 4 kohort:
- **0 - 12 Bulan:** Pelanggan Baru (Tahun Pertama)
- **13 - 24 Bulan:** Pelanggan Berkembang (Tahun Kedua)
- **25 - 48 Bulan:** Pelanggan Menengah (Tahun 3 - 4)
- **49 - 72 Bulan:** Pelanggan Loyal (Tahun 5+)


In [ ]:
cohort_summary = build_tenure_cohorts(df)
cohort_summary


> **Temuan Kohort:**
> Pelanggan baru pada tahun pertama (0-12 bulan) mengalami tingkat churn yang luar biasa tinggi (**47.44%**). 
> Begitu pelanggan melampaui 2 tahun (24 bulan), tingkat churn turun drastis ke kisaran 14-28%, dan pada kohort loyal (>4 tahun) churn hanya sebesar 9.51%.


## 2. Deteksi Titik Churn Kritis (Bulan 1 - 12)
Mari kita lihat distribusi churn pada 12 bulan pertama secara mendetail.


In [ ]:
first_year = df[df["tenure"] <= 12]
first_year_churn = first_year.groupby("tenure")["churn"].apply(lambda s: (s == "Yes").mean() * 100).reset_index()

plt.figure(figsize=(9, 4.5))
sns.barplot(data=first_year_churn, x="tenure", y="churn", color="#e74c3c")
plt.title("Tingkat Churn per Bulan pada Tahun Pertama (Bulan 0 - 12)", fontsize=13, fontweight="bold", pad=12)
plt.xlabel("Tenure (Bulan)", fontsize=11)
plt.ylabel("Churn Rate (%)", fontsize=11)
plt.tight_layout()
plt.show()


> **Insight Bulan Pertama:**
> Puncak churn terjadi pada **Bulan ke-1** (lebih dari 60% pelanggan baru yang keluar, berhenti di bulan pertama). Ini mengindikasikan adanya masalah *onboarding*, ekspektasi tagihan pertama (*bill shock*), atau kendala instalasi awal.


## 3. Matriks Bahaya Retensi (*Hazard Matrix: Tenure x Contract*)
Interaksi antara komitmen kontrak dan masa berlangganan.


In [ ]:
risk_matrix = build_risk_matrix(df, row_col="tenure", col_col="contract")
risk_matrix


In [ ]:
plt.figure(figsize=(7, 4.5))
sns.heatmap(risk_matrix, annot=True, fmt=".1f", cmap="YlOrRd", cbar_kws={'label': 'Churn Rate (%)'}, linewidths=1)
plt.title("Matriks Bahaya Retensi (Tenure Group x Contract)", fontsize=13, fontweight="bold", pad=12)
plt.xlabel("Tipe Kontrak", fontsize=11)
plt.ylabel("Kelompok Tenure", fontsize=11)
plt.tight_layout()
plt.show()


## 4. Profiling Segmen Kritis (*High-Hazard Persona*)
Berdasarkan eksplorasi, kombinasi fitur:
- Kontrak: `Month-to-month`
- Layanan Internet: `Fiber optic`
- Dukungan Teknis: `No Tech Support`
- Metode Pembayaran: `Electronic check`


In [ ]:
hazard_segment = df[
    (df["contract"] == "Month-to-month") &
    (df["internet_service"] == "Fiber optic") &
    (df["tech_support"] == "No") &
    (df["payment_method"] == "Electronic check")
]

seg_total = len(hazard_segment)
seg_churn = (hazard_segment["churn"] == "Yes").sum()
seg_churn_rate = seg_churn / seg_total * 100
seg_mrr_loss = hazard_segment[hazard_segment["churn"] == "Yes"]["monthly_charges"].sum()

print(f"Ukuran Segmen Kritis: {seg_total} pelanggan ({seg_total/len(df)*100:.1f}% dari total basis)")
print(f"Churn Rate Segmen Kritis: {seg_churn_rate:.2f}% (vs Baseline Perusahaan: 26.54%)")
print(f"MRR Loss yang Dihasilkan: ${seg_mrr_loss:,.2f} per bulan")


## Kesimpulan Analisis Kohort & Segmentasi
1. **Fokus Onboarding:** Periode 90 hari pertama adalah masa hidup-mati pelanggan di Telco. Program retensi harus dimulai sejak hari ke-1 pendaftaran.
2. **Kontrak Month-to-Month:** Kontrak bulanan pada kohort 0-12 bulan memiliki churn rate 50.8%, menjadikannya target utama insentif migrasi tahunan.
3. **Segmen Bahaya Ekstrem:** Pelanggan fiber tanpa tech support dengan tagihan cek elektronik memiliki churn rate mendekati 65%. Intervensi paket bundling tech support dan auto-pay akan memberikan dampak finansial tertinggi.
